# Notes Manager

### Note format
Every note is saved as a dictionary like this:

```python
{
    "id": 1,
    "title": "Sets in Python",
    "text": "Sets remove duplicates",
    "tags": ["python", "basics"],
    "created": "2026-10-09"
}
```

In [ ]:
# functions to help add, save and show notes
from datetime import date, datetime
from pathlib import Path
import json

path = Path("Records") / "data.json"

def add_note(notes, title, text, tags):
    """this function will create new note and add it to existing notes"""
    
    # first we will validate user data
    if title.strip() == "":
        print("title cannot be empty")
        return None
    
    # list of tags
    tags = [tag.strip() for tag in tags.split(",") if tag.strip()]
    
    # creating note
    note = {
        "id":index(notes),
        "title": title.strip(),
        "text" : text.strip(),
        "tags" : tags,
        "created" : date.today().isoformat()
    }
    
    notes.append(note)
    
    return notes
    

def save_notes(notes,path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        # convert to text first, so an error can't leave a half-written file
        content = json.dumps(notes, indent=2)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return True
            
    except TypeError:
        print("note has data that cannot be saved as json")
        return False
    except OSError:
        print("could not write the file (is it open in another program?)")
        return False
        

def load_notes(path):
    data = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            # logging
            print("data loaded successfully")
            return data
    except FileNotFoundError:
        print("file not found (load_notes)")
    except json.JSONDecodeError:
        # keep a copy of the broken file, so the next save doesn't erase old notes
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = path.with_name(f"{path.stem}_broken_{stamp}.json")
        path.replace(backup)
        print(f"file is broken, moved it to {backup}")
        
    return data
        

def index(notes):
    if len(notes) == 0:
        return 1
    else:
        return max([note["id"] for note in notes]) + 1

In [ ]:
# main function
run = True
while run:
    print("Display menu:")
    print("1) Enter note:")
    print("2) Display notes:")
    print("3) Quit:")
    
    choice = input("Enter your choice:").strip()
    
    match choice:
        case "1":
            repeat = True
            while repeat:
                notes = load_notes(path)
                title = input("Enter title:")
                text = input("Enter text:")
                tags = input("Enter tags:")
                
                notes = add_note(notes,title, text, tags)
                
                if not notes:
                    repeat = True
                else:
                    repeat = False
                    if save_notes(notes, path):
                        print(f"your note is:\n{notes[-1]}")
                    else:
                        print("note was not saved")
                    
        case "2":
            notes = load_notes(path)
            for note in notes:
                print(note["title"])
                print(note)
                
        case "3":
            run = False
        case _:
            print("invalid input")
            
            
print("program has end")

## Code review: mistakes and fixes

### 1. The wrong note was printed after saving
- **Problem:** `print(f"your note is:\n{note}")` used a variable called `note` that was never set in that part of the code. Python used the old `note` left over from another cell or the display loop, so it showed an old note.
- **Fix:** print `notes[-1]`, which is the last note in the list (the one just added).
- **Lesson:** in Jupyter, variables from earlier cells stay in memory. Code can look like it works because it uses an old variable by accident.

### 2. The empty-title check used `and`
- **Problem:** `if title == "" and tags == ""` only rejected the note when **both** were empty. A note with no title was still saved, and a title of only spaces (`"   "`) passed.
- **Fix:** `if title.strip() == ""`, print `"title cannot be empty"`, and save the stripped `title` and `text`.

### 3. Letters in the menu crashed the program
- **Problem:** `int(input(...))` raises a `ValueError` if the user types `a` or just presses Enter.
- **Fix:** keep the choice as a string and compare it to strings: `case "1":`. Also use `.strip()` so `" 1"` still works.

### 4. A wrong menu number did nothing
- **Problem:** choosing `5` just showed the menu again with no message.
- **Fix:** add `case _:` that prints `"invalid input"`.

### 5. IDs could repeat
- **Problem:** `len(notes) + 1` breaks once notes are deleted. Example: notes with ids 1, 2, 3. Delete note 1, so there are 2 notes left. The next id is `2 + 1 = 3`, which is already taken.
- **Fix:** `max(id) + 1`, so a new id is always bigger than every existing id.

### 6. A broken file would erase all notes
- **Problem:** if `data.json` was damaged, `load_notes` returned an empty list `[]`. The next save then overwrote the file with only the new note, and all old notes were lost.
- **Fix:** before continuing, `load_notes` now renames the broken file to `data_broken_<date_time>.json`, so the old data is kept.
- **Lesson:** an error while **reading** should never lead to **overwriting** good data.

### 7. `save_notes` caught the wrong errors
- **Problem:** it caught `json.JSONDecodeError`, which only happens when *reading* JSON, never when writing. The errors that can really happen while saving are `OSError` (for example `PermissionError` when the file is open in another program) and `TypeError` (data that can't be stored as JSON, like a `set`).
  Also, `open(path, "w")` empties the file straight away. If `json.dump` then fails halfway, the file is left half-written.
- **Fix:** catch `TypeError` and `OSError`, and use `json.dumps()` to turn the notes into text **before** opening the file. The main loop now also checks whether saving worked before it prints "your note is".

### 8. Smaller cleanups
- **The folder was created in two places.** Now it's created only inside `save_notes`, which is where it's needed (`parents=True` also creates any missing parent folders).
- **No encoding was given when opening files.** On Windows the default isn't UTF-8, so `encoding="utf-8"` was added to make the files read and write the same way on every computer.
- **Typo:** `"fie is broken"` was misspelled.
- **Cells 1 and 2 were code cells.** They're now markdown. The `note` dictionary in cell 2 was also the leftover variable that caused mistake 1.

---

## Who changed what

**Fixed by me after the review:** mistakes 1, 2, 4 and 5, and most of 3 (switching the menu to string choices).

**Changed by Claude:** mistakes 6, 7 and 8, plus `.strip()` on the menu choice. The exact changes, cell by cell:

#### Cell 1: title
- Changed it from a code cell holding `#notes manager` to a markdown heading: `# Notes Manager`.

#### Cell 2: note example
- Changed the `note = {...}` code cell into a markdown cell that shows the note format. This removes the global `note` variable.

#### Cell 3: functions
- **Imports:** `from datetime import date` → `from datetime import date, datetime` (`datetime` gives the time used in the backup file name).
- **Top of the cell:** removed `path.parent.mkdir(exist_ok=True)`, so the folder is only created in `save_notes`.
- **`save_notes`:**
  - `path.parent.mkdir(exist_ok= True)` → `path.parent.mkdir(parents=True, exist_ok=True)`
  - `json.dump(notes, f, indent=2)` → `content = json.dumps(notes, indent=2)`, which runs **before** the file is opened, followed by `f.write(content)`
  - `open(path, "w")` → `open(path, "w", encoding="utf-8")`
  - `return True` moved out of the `with` block
  - `except FileNotFoundError` / `except json.JSONDecodeError` → `except TypeError` / `except OSError`, with new messages
- **`load_notes`:**
  - `open(path, "r")` → `open(path, "r", encoding="utf-8")`
  - `except json.JSONDecodeError: print("fie is broken")` → renames the broken file with `path.replace(backup)` and prints where it was moved

#### Cell 4: main loop
- `choice = input("Enter your choice:")` → `input("Enter your choice:").strip()`
- `save_notes(notes, path)` → `if save_notes(notes, path):` print the note, `else:` print `"note was not saved"`
- Cleared the old output, which was from a run of the old code.

#### Cell 5: this cell
- Added this review.

**Not changed:** `add_note`, `index`, the display code (`case "2"`), the file location `Records/data.json`, and all variable and function names.

**How the changes were tested:** the code was run in a temporary folder, without touching the real `Records/data.json`. The checks covered:
- an empty title
- extra spaces in the input
- ids after a deleted note
- a save that fails
- a broken file
- the full menu with fake input: letters, a wrong number, an empty title, a valid note, display, then quit

---

### Ideas for next time
- Load the notes once at the start instead of on every loop.
- Display notes in a nicer format, and show "No notes yet" when the list is empty.
- Add features: delete a note by id, search notes by tag, edit a note.